# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [62]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [63]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./documents/ai_report_2025.pdf"
pdf_loader=PyPDFLoader(file_path)
pdf_doc=pdf_loader.load()

document_text= ""
for page in pdf_doc:
    document_text+= page.page_content + "/n"

In [64]:
document_text[:500]

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025/npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI i'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [65]:
import os
os.getenv("API_GATEWAY_KEY")

from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [66]:
from IPython.display import display, Markdown
class ReportSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
system_prompt = "You are a helpful assistant that extracts information from a report and provides a concise summary for an AI professional's development."
prompt = f"""
    Given the following context from a report, do the following:
    
    1. Find the names report's authors.
    2. Find the report's title.
    3. Identify the relevance in a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Generate a summary that is a concise and succinct no longer than 1000 tokens.
    5. Use formal academic writing style.


        
    The report is the following: 
    <report>
    {document_text}
    </report>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Relevance: <relevance statement>
    Summary: <summary>
    Tone: <tone>
"""

# Using the system prompt to provide context to the model about its role and the task at hand, which can help improve the quality of the response.
response = client.responses.create(
    model="gpt-4o",
    instructions = system_prompt,
    input = prompt
    
)

In [67]:
display(Markdown(response.output[0].content[0].text))
display(Markdown(f"### Input Tokens: {response.usage.input_tokens}"))
display(Markdown(f"### Output Tokens: {response.usage.output_tokens}"))

Author: Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Relevance: This report is highly relevant for AI professionals as it delves into the challenges and opportunities associated with implementing Generative AI in businesses. It highlights the critical factors needed to bridge the gap between pilot projects and full-scale adoption, offering insights into how AI should be developed and integrated effectively to drive tangible business outcomes.

Summary: The report, "The GenAI Divide: State of AI in Business 2025," discusses the dichotomy between high adoption rates of Generative AI (GenAI) tools and the low level of transformation observed in most enterprises. Despite significant investment, only 5% of organizations achieve measurable value. The divide is attributed not to technological limitations but to the approaches taken in AI implementation. Key findings include the preference for consumer-grade tools like ChatGPT due to their flexibility, while enterprise-level tools struggle due to poor integration and lack of adaptability. The report outlines barriers such as the GenAI systems' inability to retain context and learn over time, impacting deployment success. Successful initiatives are characterized by deep customization, learning capabilities, and integration into existing workflows. The emergence of a "shadow AI economy" demonstrates employees' preference for personal AI tools over official enterprise systems. Investment patterns reveal a heavy bias towards sales and marketing, although back-office operations often yield better ROI. The report emphasizes the importance of building adaptive, learning-capable systems and the potential of agentic AI infrastructures like NANDA and MCP to transform business processes through autonomous, interconnected systems.

Tone: Formal academic writing style.

### Input Tokens: 10900

### Output Tokens: 341

In [68]:
from pydantic import BaseModel
from openai import OpenAI
import os

class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


system_prompt = "You are a helpful assistant that extracts information from a report and provides a concise summary for an AI professional's development."
prompt = f"""
    Given the following context from a report, do the following:
    
    1. Find the names report's authors.
    2. Find the report's title.
    3. Identify the relevance in a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Generate a summary that is a concise and succinct no longer than 1000 tokens.
    5. Use formal academic writing style.


        
    The report is the following: 
    <report>
    {document_text}
    </report>
"""
completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ],
    response_format=DocumentSummary
)

completion


ParsedChatCompletion[DocumentSummary](id='chatcmpl-DZNfiMOrqaE0P0CPvPVYmMq255yR9', choices=[ParsedChoice[DocumentSummary](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[DocumentSummary](content='{"Author":"MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This report is crucial for AI professionals as it reveals the disparity between AI adoption and transformational success among businesses. Understanding the GenAI Divide will enable AI practitioners to tailor their approaches to maximize value from AI investments by emphasizing learning-capable systems, workflow integration, and strategic partnerships, thereby staying competitive in the evolving AI landscape.","Summary":"The report, \'The GenAI Divide: State of AI in Business 2025\', examines the significant gap between AI adoption and successful implementation in business environments. Despite substantial

In [69]:
summary_result = completion.choices[0].message.parsed
print("Author")
print(summary_result.Author)
print("Title")
print(summary_result.Title)
print("Relevance")
print(summary_result.Relevance)
print("Summary")
print(summary_result.Summary)
print("Tone")
print(summary_result.Tone)


Author
MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari
Title
The GenAI Divide: State of AI in Business 2025
Relevance
This report is crucial for AI professionals as it reveals the disparity between AI adoption and transformational success among businesses. Understanding the GenAI Divide will enable AI practitioners to tailor their approaches to maximize value from AI investments by emphasizing learning-capable systems, workflow integration, and strategic partnerships, thereby staying competitive in the evolving AI landscape.
Summary
The report, 'The GenAI Divide: State of AI in Business 2025', examines the significant gap between AI adoption and successful implementation in business environments. Despite substantial investment in generative AI (GenAI), only 5% of AI integrations deliver measurable value, creating what is termed as the 'GenAI Divide'. The report identifies that the problem lies not in technology quality or regulatory hurdles but in the lack of

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [87]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval import evaluate
from deepeval.models import GPTModel
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

summary_text = summary_result.Summary
test_case = LLMTestCase(
    input=document_text,                 
    actual_output=summary_text    
)


summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary capture the main ideas of the report?",
        "Is the summary concise without losing important details?",
        "Does the summary avoid unnecessary repetition?",
        "Is the summary factually consistent with the report?",
        "Does the summary provide a clear understanding of the report?"
    ],
    threshold=0.7,model=model
)

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary is logically organized, flows naturally, and is easy to understand.",
    evaluation_steps=[
        "Verify that ideas are presented in a logical order",
        "Check that transitions between ideas are smooth and natural",
        "Ensure the summary maintains consistent focus",
        "Assess whether sentences are clear and contribute to understanding",
        "Evaluate if the structure helps comprehension"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,model=model
    
)

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the tone is formal and academic.",
    evaluation_steps=[
        "Is the tone formal and professional?",
        "Does the summary avoid casual language?",
        "Is the writing appropriate for an academic audience?",
        "Does the tone remain consistent throughout?",
        "Does it reflect analytical depth?"
    ],evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,model=model
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary avoids harmful, biased, or inappropriate content.",
    evaluation_steps=[
        "Does the summary avoid harmful or unsafe content?",
        "Is the language neutral and unbiased?",
        "Does it avoid offensive or inappropriate wording?",
        "Is the content responsible and professional?",
        "Does it avoid misleading claims?"
    ],evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.9,model=model
)



results = evaluate(
    test_cases=[test_case],
    metrics=[
        summarization_metric,
        coherence_metric,
        tonality_metric,
        safety_metric
    ]
)






✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Summarization (score: 0.46153846153846156, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.46 because the summary contains significant contradictions to the original text regarding the value of AI integrations and the nature of successful organizations' systems. Additionally, it introduces several pieces of extra information that were not present in the original text, which further detracts from its accuracy and relevance., error: None)
  - ✅ Coherence [GEval] (score: 0.8024475174739207, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response presents ideas in a logical order, starting with the introduction of the report and moving through the key findings and implications. Transitions between ideas are mostly smooth, although some sections could benefit from clearer connections. The summary maintains a consistent focus on the challenges and opportunities in AI adoption. Sentences are generally cle

✓ Evaluation completed 🎉! (time taken: 20.66s | token cost: 0.00484305 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [98]:
client_enhanced = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)


enhancement_system_prompt = """You are a helpful AI assistant specializing in document analysis and summarization.
Your task is to create an improved summary based on evaluation feedback, focusing on accuracy and faithfulness to the source material."""

enhancement_user_prompt = f"""
You previously created a summary that received the following evaluation scores:
- Summarization Score: {evaluation_results['SummarizationScore']}
- Coherence Score: {evaluation_results['CoherenceScore']}
- Tonality Score: {evaluation_results['TonalityScore']}
- Safety Score: {evaluation_results['SafetyScore']}

EVALUATION FEEDBACK:
{evaluation_results['SummarizationReason']}

ORIGINAL DOCUMENT:
{document_text}

PREVIOUS SUMMARY:
{summary_result.Summary}

INSTRUCTIONS FOR IMPROVEMENT:
1. Create a new summary that ONLY includes information explicitly stated in the original document
2. Do not add interpretations, inferences, or extra information beyond what's in the source
3. Maintain the {TONE} style
4. Ensure the summary is concise (maximum 1000 tokens)
5. Focus on accuracy and faithfulness to the source material
6. Keep the same structure (Author, Title, Relevance, Summary, Tone)

Generate an enhanced summary that addresses the evaluation feedback.
"""


enhanced_completion = client_enhanced.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": enhancement_system_prompt},
        {"role": "user", "content": enhancement_user_prompt}
    ],
    response_format=DocumentSummary
)

enhanced_summary = enhanced_completion.choices[0].message.parsed

print("Enhanced Summary Results:")
print(f"\nAuthor: {enhanced_summary.Author}")
print(f"\nTitle: {enhanced_summary.Title}")
print(f"\nRelevance:\n{enhanced_summary.Relevance}")
print(f"\nSummary:\n{enhanced_summary.Summary}")
print(f"\nTone: {enhanced_summary.Tone}")



Enhanced Summary Results:

Author: MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Relevance:
The report is relevant to businesses seeking to enhance AI implementation strategies and achieve meaningful GenAI-driven transformation.

Summary:
The report examines the disparity between AI adoption and effective utilization, termed the 'GenAI Divide.' Despite extensive investments ranging from $30-40 billion in generative AI, only 5% of AI initiatives provide a measurable return on investment, mainly due to shortcomings in learning, integration, and contextual adaptation of AI systems. Tools like ChatGPT are widely adopted but fail to scale effectively within enterprises due to issues such as brittle workflows and inadequate contextual learning. The report identifies four patterns characterizing the divide: limited disruption in industries, mismatched investments prioritizing top-line functions, ineffective i

In [100]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval import evaluate
from deepeval.models import GPTModel
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

summary_text = summary_result.Summary
test_case = LLMTestCase(
    input=document_text,                 
    actual_output=enhanced_summary.Summary    
)


summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary capture the main ideas of the report?",
        "Is the summary concise without losing important details?",
        "Does the summary avoid unnecessary repetition?",
        "Is the summary factually consistent with the report?",
        "Does the summary provide a clear understanding of the report?"
    ],
    threshold=0.7,model=model
)

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary is logically organized, flows naturally, and is easy to understand.",
    evaluation_steps=[
        "Verify that ideas are presented in a logical order",
        "Check that transitions between ideas are smooth and natural",
        "Ensure the summary maintains consistent focus",
        "Assess whether sentences are clear and contribute to understanding",
        "Evaluate if the structure helps comprehension"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,model=model
    
)

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the tone is formal and academic.",
    evaluation_steps=[
        "Is the tone formal and professional?",
        "Does the summary avoid casual language?",
        "Is the writing appropriate for an academic audience?",
        "Does the tone remain consistent throughout?",
        "Does it reflect analytical depth?"
    ],evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,model=model
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary avoids harmful, biased, or inappropriate content.",
    evaluation_steps=[
        "Does the summary avoid harmful or unsafe content?",
        "Is the language neutral and unbiased?",
        "Does it avoid offensive or inappropriate wording?",
        "Is the content responsible and professional?",
        "Does it avoid misleading claims?"
    ],evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.8,model=model
)



results = evaluate(
    test_cases=[test_case],
    metrics=[
        summarization_metric,
        coherence_metric,
        tonality_metric,
        safety_metric
    ]
)






✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Summarization (score: 0.4666666666666667, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.47 because the summary contains significant contradictions to the original text, such as misrepresenting the patterns of the GenAI Divide and introducing concepts not found in the original, like the 'shadow AI economy' and 'Agentic Web'. Additionally, it includes extra information that diverges from the original content, which detracts from the overall accuracy and reliability of the summary., error: None)
  - ✅ Coherence [GEval] (score: 0.7950000723701834, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response presents ideas in a logical order, starting with the definition of the 'GenAI Divide' and moving through various factors contributing to it. Transitions between ideas are mostly smooth, although some sections could benefit from clearer connections. The summary maintains a consistent focus on the challe

✓ Evaluation completed 🎉! (time taken: 21.07s | token cost: 0.0049108500000000005 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

When adding greater control over the summary has led the model to hallucinate.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
